# AI-Driven Digital Twin for Water Distribution Networks
## Chapter 5: Machine Learning Leak Detection & Localization Pipeline

**Team:** Azim Abdulla, Adithya Shaji, Noel John  
**Supervisor:** Dr. Archana T | School of Computer Science & Engineering (SCOPE), VIT  

---

### Objective
This notebook demonstrates the end-to-end Machine Learning pipeline for the Water Distribution Network (WDN) Digital Twin:
1. **Hydraulic Simulation & Data Generation** using EPANET and WNTR.
2. **Exploratory Data Analysis (EDA)** on sensor pressure deviations.
3. **Random Forest Ensemble Training** across 10 distinct operational classes (Normal + 9 leak junctions).
4. **Model Evaluation**, Confusion Matrix Heatmaps, and Sensor Feature Importance Analysis.

### 1. Setup & Library Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import wntr

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set visual style
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Paths
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd(), 'data')
INP_FILE = os.path.join(DATA_DIR, 'Net1.inp')
DATASET_FILE = os.path.join(DATA_DIR, 'leak_dataset.csv')
MODEL_FILE = os.path.join(DATA_DIR, 'leak_model.joblib')

JUNCTIONS = ['10', '11', '12', '13', '21', '22', '23', '31', '32']
print("Environment initialized. Target network: Net1 (9 Sensor Junctions)")

### 2. Physical Water Network Model (EPANET Net1)
Load the EPANET model and inspect node topology and baseline hydraulics.

In [ ]:
wn = wntr.network.WaterNetworkModel(INP_FILE)
print(f"Total Nodes: {wn.num_nodes} (Junctions: {wn.num_junctions}, Tanks: {wn.num_tanks}, Reservoirs: {wn.num_reservoirs})")
print(f"Total Links: {wn.num_links} (Pipes: {wn.num_pipes}, Pumps: {wn.num_pumps})")

# Run baseline 24-hour simulation
wn.options.time.duration = 24 * 3600
wn.options.hydraulic.demand_model = 'PDD'
sim = wntr.sim.WNTRSimulator(wn)
baseline_results = sim.run_sim()

print("Baseline 24-Hour simulation succeeded.")

### 3. Load Synthetic Hydraulic Dataset (5,000 Samples)
The dataset contains 5,000 simulation instances across diverse demand multipliers ($0.7\times - 1.3\times$), leak sizes ($0.001 - 0.015\text{ m}^2$), and sensor Gaussian noise.

In [ ]:
if not os.path.exists(DATASET_FILE):
    from leak_detection.dataset_generator import generate_synthetic_dataset
    df = generate_synthetic_dataset(num_samples=5000, output_csv=DATASET_FILE)
else:
    df = pd.read_csv(DATASET_FILE)

print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution across 10 Operational States:")
print(df['leak_node'].value_counts())
df.head()

### 4. Exploratory Data Analysis (EDA)
Visualize the hydraulic disturbance and pressure drop signature caused by a leak at Junction 11.

In [ ]:
# Plot pressure comparison across sensor junctions for Normal vs Leak at J11
normal_sample = df[df['leak_node'] == 'Normal'].iloc[0]
leak_sample_j11 = df[df['leak_node'] == '11'].iloc[0]

p_cols = [f'P_{j}' for j in JUNCTIONS]
labels = [f'J{j}' for j in JUNCTIONS]

plt.figure(figsize=(11, 5))
x = np.arange(len(JUNCTIONS))
width = 0.35

plt.bar(x - width/2, [normal_sample[col] for col in p_cols], width, label='Normal Operating State', color='#0284c7')
plt.bar(x + width/2, [leak_sample_j11[col] for col in p_cols], width, label='Leak State (at J11)', color='#ef4444')

plt.xlabel('Sensor Junctions')
plt.ylabel('Pressure Head (meters)')
plt.title('Hydraulic Pressure Profile: Normal vs. Active Pipe Leak at Junction 11')
plt.xticks(x, labels)
plt.legend()
plt.tight_layout()
plt.show()

### 5. Model Training: Random Forest Classifier
We train a 100-tree Random Forest Classifier using 18 sensor features ($[P_{10..32}, \Delta P_{10..32}]$).

In [ ]:
feature_cols = [f'P_{j}' for j in JUNCTIONS] + [f'dP_{j}' for j in JUNCTIONS]
X = df[feature_cols]
y = df['leak_node'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Random Forest Test Accuracy: {test_acc * 100:.2f}%")

### 6. Performance Evaluation: Confusion Matrix Heatmap

In [ ]:
classes = clf.classes_
cm = confusion_matrix(y_test, y_pred, labels=classes)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Leak Localization Node')
plt.ylabel('Actual Ground Truth Node')
plt.title(f'Confusion Matrix Heatmap (Accuracy: {test_acc * 100:.1f}%)')
plt.tight_layout()
plt.show()

print("Detailed Classification Metrics:\n")
print(classification_report(y_test, y_pred, target_names=classes))

### 7. Feature Importance Analysis (Sensor Sensitivity Ranking)
Evaluate which pipe junctions provide the strongest hydraulic gradient signals for leak detection.

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#38bdf8')
plt.title('Random Forest Feature Importance: Sensor Sensitivity Ranking')
plt.xlabel('Gini Importance Score')
plt.ylabel('Sensor Hydraulic Features')
plt.tight_layout()
plt.show()

### 8. Real-Time Inference Verification

In [ ]:
# Test inference on a simulated leak vector
test_pressures = {'10': 89.0, '11': 36.2, '12': 58.4, '13': 62.0, '21': 80.0, '22': 75.0, '23': 70.0, '31': 82.0, '32': 80.0}
test_baseline = {'10': 89.7, '11': 88.9, '12': 87.2, '13': 85.1, '21': 82.3, '22': 80.4, '23': 78.1, '31': 82.0, '32': 80.5}

from leak_detection.inference import predict_leak
result = predict_leak(test_pressures, test_baseline)

print("\n--- Live ML Prediction Output ---")
print(f"Leak Detected: {result['is_leak']}")
print(f"Leak Probability: {result['leak_probability'] * 100:.1f}%")
print(f"Localized Junction: Junction {result['localized_node']}")
print(f"Severity: {result['severity']}")
print(f"Model Type: {result['model_type']}")
print(f"Top Affected Sensors: {result['top_affected_nodes']}")